In [0]:
from pyspark.sql.functions import *
import json

In [0]:
dbutils.widgets.text("catalog_name", "finalproject")

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')

In [0]:
df_compras = spark.table(f'{catalog_name}.silver.compras')

df_detalles = spark.table(f'{catalog_name}.silver.detalles')

## Implementando reglas de calidad

In [0]:
reglas = []

### Reglas de calidad en df_compras

In [0]:
# 1. No null en fecha_envio
nulos_fecha_envio = df_compras.filter((col('estado') == 'Entregado') & (col('fecha_envio').isNull())).count()
reglas.append({
    'tabla': 'df_compras',
    'columna': 'fecha_envio',
    'regla': 'no_null',
    'cumple': nulos_fecha_envio == 0,
    'detalle': f"Registros nulos: {nulos_fecha_envio}"
})

# 2.Fecha_envio mayor o igual a Fecha_orden
nulos_envio_orden = df_compras.filter( (col('estado') == 'Entregado') & ( col('fecha_envio') < col('fecha_orden') ) ).count()
reglas.append({
    'tabla': 'df_compras',
    'columna': 'fecha_envio',
    'regla': 'Fecha_envio después de la fecha_orden',
    'cumple': nulos_envio_orden == 0,
    'detalle': f"Registros nulos: {nulos_envio_orden}"
})

### Reglas de calidad en df_detalles

In [0]:
# 1.Subtotal mayor a 0
nulos_envio_orden = df_detalles.filter( col('subtotal') <= 0 ).count()
reglas.append({
    'tabla': 'df_detalles',
    'columna': 'subtotal',
    'regla': 'mayor a 0',
    'cumple': nulos_envio_orden == 0,
    'detalle': f"Subtotales menores o iguales a 0: {nulos_envio_orden}"
})

## Ingesta de datos de las reglas

In [0]:
df_reglas = spark.createDataFrame(reglas)
df_reglas = df_reglas.withColumn('fecha_validacion', current_timestamp())

### Creamos la tabla donde insertaremos los `datos`

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {catalog_name}.gold.log_calidad_datos
(
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  tabla STRING,
  columna STRING,
  regla STRING,
  cumple BOOLEAN,
  detalle STRING,
  fecha_validacion TIMESTAMP
)
          """)

### Insertamos datos en la tabla log_calidad_datos

In [0]:
(
    df_reglas.write\
        .mode('append')\
        .format('delta')\
        .saveAsTable(f'{catalog_name}.log_calidad_datos')
)

In [0]:
fallas_criticas = [regla for regla in reglas if not regla['cumple']]

if fallas_criticas:
    dbutils.jobs.taskValues.set(key = 'estado', values = 'Falla Critica')
    dbutils.jobs.taskValues.set(key = 'detalle', values = json.dumps(fallas_criticas))
else:
    dbutils.jobs.taskValues.set(key = 'estado', value = 'Validación Exitosa')